In [6]:
from pathlib import Path
import sys
from itertools import product

import numpy as np
import torch

# Find the project root from the notebook's working directory.
current = Path.cwd().resolve()
root = next(
    (
        folder
        for folder in (current, *current.parents)
        if (folder / "inner maximization" / "NonrectangularSolver.py").is_file()
    ),
    None,
)

if root is None:
    raise FileNotFoundError("Run this notebook from inside your awdro folder.")

sys.path.insert(0, str(root / "inner maximization"))

from NonrectangularSolver import NonrectangularSolver

In [7]:
class ToyModel:
    s0 = 2.0
    v0 = 0.0

    def is_valid_state(self, state):
        state = np.asarray(state, dtype=float)

        return (
            state.shape == (2,)
            and np.isfinite(state).all()
            and state[0] > 0
            and state[1] >= 0
        )

    def step(self, state, dt, rng):
        # This fixture is designed for exactly two branches.
        if state.shape != (2, 2):
            raise ValueError("Use reference_branches=2 for this test.")

        # Root state is 2; first-period states are 1 and 3.
        if state[0, 0] == self.s0:
            stocks = [1.0, 3.0]
        else:
            stocks = [1.0, 9.0]

        return np.column_stack([stocks, np.zeros(2)])


def make_solver(budget):
    return NonrectangularSolver(
        reference_model=ToyModel(),
        trading_times=[0.0, 0.5, 1.0],
        budget=budget,
        discretization={
            "reference_branches": 2,
            "seed": 42,
        },
    )


def constant_loss(paths):
    return torch.full(
        (paths.shape[0],),
        7.0,
        dtype=paths.dtype,
        device=paths.device,
    )


def matching_loss(paths):
    # Reward 10 when the first and second stock observations
    # are both in their respective low states or both high.
    first_high = paths[:, 1, 0] > 2.0
    second_high = paths[:, 2, 0] > 5.0

    return 10.0 * (first_high == second_high).to(paths.dtype)

In [8]:
budgets = [0.0, 0.5, 1.0, 2.0, 4.0, 6.0]
computed_values = []

for budget in budgets:
    solver = make_solver(budget)

    # A constant loss cannot be increased by changing the distribution.
    constant_result = solver.solve(constant_loss)
    np.testing.assert_allclose(
        constant_result["value"],
        7.0,
        rtol=0.0,
        atol=1e-10,
    )

    # Compare against the analytical answer.
    result = solver.solve(matching_loss)
    expected = min(5.0 + (10.0 / 64.0) * budget**2, 10.0)

    np.testing.assert_allclose(
        result["value"],
        expected,
        rtol=0.0,
        atol=2e-5,  # Allow for scalar-optimizer numerical error.
    )

    if budget == 0:
        assert result["multiplier"] is None

    computed_values.append(result["value"])

    print(
        f"budget={budget:4.1f}  "
        f"computed={result['value']:.8f}  "
        f"expected={expected:.8f}"
    )

# Larger ambiguity sets must not decrease the worst-case value.
assert np.all(np.diff(computed_values) >= -2e-5)

print("Constant-loss, zero-budget, and robust-value checks passed.")

budget= 0.0  computed=5.00000000  expected=5.00000000
budget= 0.5  computed=5.03906257  expected=5.03906250
budget= 1.0  computed=5.15625003  expected=5.15625000
budget= 2.0  computed=5.62500005  expected=5.62500000
budget= 4.0  computed=7.50000103  expected=7.50000000
budget= 6.0  computed=10.00000000  expected=10.00000000
Constant-loss, zero-budget, and robust-value checks passed.


In [9]:
solver = make_solver(budget=0.0)

# Budget zero makes dual_objective equal the penalized DP value U(lambda).
# Calling dual_objective directly does not use solve's zero-budget shortcut.
causal_value = solver.dual_objective(
    multiplier=1.0,
    loss_fn=matching_loss,
)

# Independent benchmark: allow choices to depend on the entire source path.
anticipative_value = 0.0

for x1, x2 in product([1.0, 3.0], [1.0, 9.0]):
    best_value = -np.inf

    for y1, y2 in product([1.0, 3.0], [1.0, 9.0]):
        reward = 10.0 * ((y1 > 2.0) == (y2 > 5.0))
        transport_cost = (x1 - y1)**2 + (x2 - y2)**2

        best_value = max(best_value, reward - transport_cost)

    # Each of the four reference paths has probability 1/4.
    anticipative_value += 0.25 * best_value

np.testing.assert_allclose(causal_value, 5.0, atol=1e-10, rtol=0.0)
np.testing.assert_allclose(anticipative_value, 8.0, atol=1e-10, rtol=0.0)

assert causal_value < anticipative_value

print(f"Causal penalized value:       {causal_value:.6f}")
print(f"Anticipative penalized value: {anticipative_value:.6f}")
print("Causality check passed.")

Causal penalized value:       5.000000
Anticipative penalized value: 8.000000
Causality check passed.


In [10]:
class VarianceToyModel:
    s0 = 2.0
    v0 = 0.0

    def is_valid_state(self, state):
        state = np.asarray(state, dtype=float)

        return (
            state.shape == (2,)
            and np.isfinite(state).all()
            and state[0] > 0
            and state[1] >= 0
        )

    def step(self, state, dt, rng):
        if state.shape != (2, 2):
            raise ValueError("Use reference_branches=2.")

        return np.array([
            [2.0, 0.0],
            [2.0, 1.0],
        ])


def variance_loss(paths):
    return paths[:, -1, 1]


for budget in [0.0, 0.25, 0.5, 1.0]:
    solver = NonrectangularSolver(
        reference_model=VarianceToyModel(),
        trading_times=[0.0, 1.0],
        budget=budget,
        discretization={
            "reference_branches": 2,
            "seed": 42,
        },
    )

    result = solver.solve(variance_loss)
    expected = min(0.5 + budget**2, 1.0)

    np.testing.assert_allclose(
        result["value"],
        expected,
        rtol=0.0,
        atol=2e-5,
    )

    print(
        f"budget={budget:.2f}  "
        f"computed={result['value']:.8f}  "
        f"expected={expected:.8f}"
    )

print("Variance-perturbation check passed.")

budget=0.00  computed=0.50000000  expected=0.50000000
budget=0.25  computed=0.56250001  expected=0.56250000
budget=0.50  computed=0.75000000  expected=0.75000000
budget=1.00  computed=1.00000000  expected=1.00000000
Variance-perturbation check passed.


In [11]:
sys.path.insert(0, str(root / "path simulation"))

from Heston import Heston

model = Heston(
    s0=100.0,
    v0=0.04,
    mu=0.0,
    kappa=2.0,
    theta=0.04,
    vol_of_vol=0.3,
    rho=-0.7,
)


def call_loss(paths):
    # Also equals shortfall for a call with zero capital and zero holdings.
    return torch.clamp(paths[:, -1, 0] - 100.0, min=0.0)


def make_heston_solver(budget):
    return NonrectangularSolver(
        reference_model=model,
        trading_times=[0.0, 0.5, 1.0],
        budget=budget,
        discretization={
            "reference_branches": 2,
            "seed": 42,
        },
    )


solver = make_heston_solver(0.0)

In [12]:
nodes = solver.reference_nodes

# Record each child's parent and unconditional probability.
parents = {}
node_probabilities = {0: 1.0}

for parent_id, node in enumerate(nodes):
    for child_id, probability in zip(
        node["children"], node["probabilities"]
    ):
        parents[child_id] = parent_id
        node_probabilities[child_id] = (
            node_probabilities[parent_id] * probability
        )

leaf_paths = []
leaf_probabilities = []

for node_id, node in enumerate(nodes):
    if node["children"]:
        continue

    history = []
    current_id = node_id

    while True:
        history.append(nodes[current_id]["state"])

        if current_id == 0:
            break

        current_id = parents[current_id]

    leaf_paths.append(history[::-1])
    leaf_probabilities.append(node_probabilities[node_id])

leaf_paths = np.asarray(leaf_paths, dtype=float)
leaf_probabilities = np.asarray(leaf_probabilities, dtype=float)

np.testing.assert_allclose(
    leaf_probabilities.sum(), 1.0, rtol=0.0, atol=1e-12
)

assert leaf_paths.shape == (4, 3, 2)
assert np.isfinite(leaf_paths).all()
assert np.all(leaf_paths[:, :, 0] > 0)
assert np.all(leaf_paths[:, :, 1] >= 0)

# Compute this benchmark directly in NumPy.
leaf_losses = np.maximum(leaf_paths[:, -1, 0] - 100.0, 0.0)
nominal_expected = float(leaf_probabilities @ leaf_losses)

zero_budget_result = solver.solve(call_loss)

np.testing.assert_allclose(
    zero_budget_result["value"],
    nominal_expected,
    rtol=0.0,
    atol=1e-10,
)

print(f"Nominal expectation: {nominal_expected:.8f}")
print("Zero-budget check passed.")

Nominal expectation: 10.22151148
Zero-budget check passed.


In [13]:
budgets = [0.0, 0.5, 1.0, 2.0]
robust_values = []
tolerance = 1e-4

reference_states = np.asarray([
    node["state"] for node in solver.reference_nodes
])

# For this terminal-only payoff, this is the maximum available loss.
terminal_candidates = solver.adversarial_candidates[2]
maximum_loss = float(
    np.maximum(terminal_candidates[:, 0] - 100.0, 0.0).max()
)

for budget in budgets:
    candidate_solver = make_heston_solver(budget)

    # A budget comparison must use the same reference discretization.
    candidate_states = np.asarray([
        node["state"] for node in candidate_solver.reference_nodes
    ])
    np.testing.assert_array_equal(candidate_states, reference_states)

    result = candidate_solver.solve(call_loss)
    value = float(result["value"])

    assert np.isfinite(value)
    assert value >= nominal_expected - tolerance
    assert value <= maximum_loss + tolerance

    if budget > 0:
        assert np.isfinite(result["multiplier"])
        assert result["multiplier"] >= 0

    robust_values.append(value)

    print(
        f"budget={budget:.2f}  "
        f"value={value:.8f}  "
        f"multiplier={result['multiplier']}"
    )

assert np.all(np.diff(robust_values) >= -tolerance)

print("Heston-tree bounds and budget-monotonicity checks passed.")

budget=0.00  value=10.22151148  multiplier=None
budget=0.50  value=10.31453576  multiplier=0.046865740977597735
budget=1.00  value=10.34968507  multiplier=0.046865740977597735
budget=2.00  value=10.49028241  multiplier=0.04686577203651596
Heston-tree bounds and budget-monotonicity checks passed.


In [14]:
sys.path.insert(0, str(root / "hedging"))

from SuperhedgingProblem import SuperhedgingProblem


def call_payoff(paths):
    return torch.clamp(paths[:, -1, 0] - 100.0, min=0.0)


def constant_hedge(t, history):
    # Hold half a share throughout the trading period.
    return torch.full(
        (history.shape[0],),
        0.5,
        dtype=history.dtype,
        device=history.device,
    )


problem = SuperhedgingProblem(
    payoff=call_payoff,
    trading_times=np.array([0.0, 0.5, 1.0]),
)

initial_capital = 5.0


def hedging_loss(paths):
    return problem.shortfall(
        paths,
        initial_capital,
        constant_hedge,
    )

In [15]:
paths_tensor = torch.tensor(leaf_paths, dtype=torch.float64)

actual_losses = hedging_loss(paths_tensor).detach().cpu().numpy()

terminal_stock = leaf_paths[:, -1, 0]
initial_stock = leaf_paths[:, 0, 0]

expected_gains = 0.5 * (terminal_stock - initial_stock)
expected_payoffs = np.maximum(terminal_stock - 100.0, 0.0)

expected_losses = np.maximum(
    expected_payoffs - initial_capital - expected_gains,
    0.0,
)

np.testing.assert_allclose(
    actual_losses,
    expected_losses,
    rtol=0.0,
    atol=1e-10,
)

nominal_shortfall = float(leaf_probabilities @ expected_losses)

zero_result = make_heston_solver(0.0).solve(hedging_loss)

np.testing.assert_allclose(
    zero_result["value"],
    nominal_shortfall,
    rtol=0.0,
    atol=1e-10,
)

print(f"Nominal expected shortfall: {nominal_shortfall:.8f}")
print("Hedging-loss and zero-budget checks passed.")

Nominal expected shortfall: 2.55671582
Hedging-loss and zero-budget checks passed.


In [16]:
paths_tensor = torch.tensor(leaf_paths, dtype=torch.float64)

actual_losses = hedging_loss(paths_tensor).detach().cpu().numpy()

terminal_stock = leaf_paths[:, -1, 0]
initial_stock = leaf_paths[:, 0, 0]

expected_gains = 0.5 * (terminal_stock - initial_stock)
expected_payoffs = np.maximum(terminal_stock - 100.0, 0.0)

expected_losses = np.maximum(
    expected_payoffs - initial_capital - expected_gains,
    0.0,
)

np.testing.assert_allclose(
    actual_losses,
    expected_losses,
    rtol=0.0,
    atol=1e-10,
)

nominal_shortfall = float(leaf_probabilities @ expected_losses)

zero_result = make_heston_solver(0.0).solve(hedging_loss)

np.testing.assert_allclose(
    zero_result["value"],
    nominal_shortfall,
    rtol=0.0,
    atol=1e-10,
)

print(f"Nominal expected shortfall: {nominal_shortfall:.8f}")
print("Hedging-loss and zero-budget checks passed.")

Nominal expected shortfall: 2.55671582
Hedging-loss and zero-budget checks passed.


In [17]:
def threshold_hedge(t, history):
    current_stock = history[:, -1, 0]
    return (current_stock < 100.0).to(history.dtype)


def adaptive_hedging_loss(paths):
    return problem.shortfall(
        paths,
        initial_capital,
        threshold_hedge,
    )

In [18]:
stocks = leaf_paths[:, :, 0]

# Holdings use the stock price BEFORE each price increment.
holdings = (stocks[:, :-1] < 100.0).astype(float)
increments = np.diff(stocks, axis=1)

expected_gains = (holdings * increments).sum(axis=1)
expected_losses = np.maximum(
    np.maximum(stocks[:, -1] - 100.0, 0.0)
    - initial_capital
    - expected_gains,
    0.0,
)

actual_losses = (
    adaptive_hedging_loss(paths_tensor)
    .detach()
    .cpu()
    .numpy()
)

np.testing.assert_allclose(
    actual_losses, expected_losses, rtol=0.0, atol=1e-10
)

nominal_expected = float(leaf_probabilities @ expected_losses)
result = make_heston_solver(0.0).solve(adaptive_hedging_loss)

np.testing.assert_allclose(
    result["value"], nominal_expected, rtol=0.0, atol=1e-10
)

print("History-dependent hedge checks passed.")

History-dependent hedge checks passed.


In [19]:
from itertools import product
from scipy.optimize import linprog


def finite_causal_lp(
    reference_paths,
    probabilities,
    candidates,
    budget,
    loss_fn,
):
    """Compute the causal robust expectation on a small finite path space."""
    x = np.asarray(reference_paths, dtype=float)
    p = np.asarray(probabilities, dtype=float)

    n, n_times, n_features = x.shape

    assert p.shape == (n,)
    assert np.all(p > 0)
    np.testing.assert_allclose(p.sum(), 1.0)

    # This benchmark assumes the same fixed initial state on every path.
    np.testing.assert_array_equal(
        x[:, 0, :],
        np.tile(x[0, 0], (n, 1)),
    )

    # Enumerate all adversarial paths allowed by the DP's candidate sets.
    y = np.asarray([
        np.vstack([x[0, 0], *states])
        for states in product(
            *(candidates[t] for t in range(1, n_times))
        )
    ])

    m = len(y)

    with torch.no_grad():
        losses = loss_fn(
            torch.tensor(y, dtype=torch.float64)
        ).cpu().numpy()

    assert losses.shape == (m,)
    assert np.isfinite(losses).all()

    # Whole-path squared Euclidean transport costs.
    costs = (
        (x[:, None, :, :] - y[None, :, :, :])**2
    ).sum(axis=(2, 3))

    equality_rows = []
    equality_values = []

    # Reference marginal: sum_j pi[i, j] = p[i].
    for i in range(n):
        row = np.zeros((n, m))
        row[i, :] = 1.0

        equality_rows.append(row.ravel())
        equality_values.append(p[i])

    # Causality constraints at each intermediate date.
    # Initial states are fixed; at the terminal date no further
    # nonanticipation restriction is needed.
    for t in range(1, n_times - 1):
        source_prefixes = x[:, :t + 1, :].reshape(n, -1)
        target_prefixes = y[:, :t + 1, :].reshape(m, -1)

        _, source_groups = np.unique(
            source_prefixes,
            axis=0,
            return_inverse=True,
        )
        _, target_groups = np.unique(
            target_prefixes,
            axis=0,
            return_inverse=True,
        )

        for source_group in np.unique(source_groups):
            indices = np.flatnonzero(source_groups == source_group)
            anchor = indices[0]

            for i in indices[1:]:
                for target_group in np.unique(target_groups):
                    mask = target_groups == target_group
                    row = np.zeros((n, m))

                    # Conditional probabilities of each target prefix
                    # must agree across source paths sharing a prefix.
                    row[i, mask] = 1.0 / p[i]
                    row[anchor, mask] = -1.0 / p[anchor]

                    equality_rows.append(row.ravel())
                    equality_values.append(0.0)

    result = linprog(
        # SciPy minimizes, so negate the expected loss.
        c=-np.tile(losses, n),

        # Global expected transport cost <= budget squared.
        A_ub=costs.reshape(1, -1),
        b_ub=[budget**2],

        A_eq=np.asarray(equality_rows),
        b_eq=np.asarray(equality_values),

        # Joint probabilities must be nonnegative.
        bounds=(0.0, None),
        method="highs",
    )

    if not result.success:
        raise RuntimeError(f"Causal LP failed: {result.message}")

    return -float(result.fun)

In [20]:
for budget in [0.0, 0.5, 1.0, 2.0]:
    solver = make_heston_solver(budget)

    # Confirm that the supplied reference paths belong to this tree.
    current_leaf_paths = []

    for first_id in solver.reference_nodes[0]["children"]:
        for second_id in solver.reference_nodes[first_id]["children"]:
            current_leaf_paths.append([
                solver.reference_nodes[node_id]["state"]
                for node_id in [0, first_id, second_id]
            ])

    np.testing.assert_array_equal(
        np.asarray(current_leaf_paths),
        leaf_paths,
    )

    dp_value = float(solver.solve(adaptive_hedging_loss)["value"])

    lp_value = finite_causal_lp(
        reference_paths=leaf_paths,
        probabilities=leaf_probabilities,
        candidates=solver.adversarial_candidates,
        budget=budget,
        loss_fn=adaptive_hedging_loss,
    )

    np.testing.assert_allclose(
        dp_value,
        lp_value,
        rtol=0.0,
        atol=1e-4,
    )

    print(
        f"budget={budget:.2f}  "
        f"DP={dp_value:.8f}  "
        f"LP={lp_value:.8f}  "
        f"gap={abs(dp_value - lp_value):.2e}"
    )

print("Independent causal LP comparison passed.")

budget=0.00  DP=6.47151148  LP=6.47151148  gap=0.00e+00
budget=0.50  DP=6.56298104  LP=6.56298091  gap=1.36e-07
budget=1.00  DP=6.59289484  LP=6.59289424  gap=5.94e-07
budget=2.00  DP=6.71255002  LP=6.71254760  gap=2.42e-06
Independent causal LP comparison passed.
